## Log and register model: PyFunc XGBoost

### Purpose
This notebook ingests a custom PyFunc model (XGBoost or compatible with pre/post config) trained outside Databricks,
logs it to MLflow as a PyFunc, and registers it in Unity Catalog for governed deployment.

Use this notebook when:
- The customer provides a model plus preprocessing and postprocessing config (e.g. centering, scaling, output format)
- You need a single wrapper that applies pre/post logic at predict time
- You want to log with `mlflow.pyfunc` for consistent serving and batch inference

This is Stage 2 of the BYOM workflow (Log & Register).

### Expected inputs
Artifacts must already exist in a Unity Catalog volume and follow the expected layout defined in Stage 1.

Required:
- Model file (e.g. `pyfunc_xgb.pkl`)
- Preprocessing config (e.g. `pyfunc_preproc.json`)
- Postprocessing config (e.g. `pyfunc_postproc.json`)

Optional but recommended:
- `canonical_input.json`
- `checksums.json` for artifact integrity validation


In [ ]:
%pip install -q threadpoolctl>=3.5.0 xgboost torch
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [ ]:
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("volume_name", "volume", "Volume Name")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
volume = dbutils.widgets.get("volume_name")

### Load Artifacts from Unity Catalog Volume

Reads model artifacts from the configured UC volume using notebook widgets:

- `catalog_name`
- `schema_name`
- `volume_name`

Ensure these are aligned before running.

If `checksums.json` is present:
- Verify artifact integrity before logging
- Fail fast if mismatches are detected

Recommended for production engagements.


In [ ]:
%run ./00_shared

In [ ]:
import tempfile
from types import SimpleNamespace
import json
import joblib
import numpy as np
import pandas as pd
import mlflow
from mlflow import pyfunc
from mlflow.tracking import MlflowClient


# PyFunc wrapper: loads XGB/sklearn model + pre/post config from artifacts; applies preprocessing before predict and postprocessing after.
class XGBPyFunc(pyfunc.PythonModel):
    """PyFunc wrapper for XGBoost (or sklearn) with pre/post config. Requires artifact 'feature_columns' (JSON list)."""

    def load_context(self, context):
        self.model = joblib.load(context.artifacts["xgb_model"])
        with open(context.artifacts["preproc"], "r", encoding="utf-8") as f:
            self.pre = json.load(f)
        with open(context.artifacts["postproc"], "r", encoding="utf-8") as f:
            self.post = json.load(f)
        # PythonModelContext only has .artifacts; persist feature_columns as an artifact
        with open(context.artifacts["feature_columns"], "r", encoding="utf-8") as f:
            self.feature_columns = json.load(f)

    def _pre(self, df: pd.DataFrame) -> np.ndarray:
        if list(df.columns) != self.feature_columns or df.shape[0] < 1:
            raise ValueError("Input schema/rows mismatch.")
        x = df[self.feature_columns].to_numpy(dtype=np.float32)
        if self.pre.get("center", False):
            x = x - np.array(self.pre["means"], dtype=np.float32)
        if self.pre.get("scale", False):
            x = x / np.array(self.pre["stds"], dtype=np.float32)
        return x

    def _post(self, raw) -> pd.DataFrame:
        y = np.asarray(raw)
        if self.post.get("output") == "proba" and y.ndim == 2:
            y = y.max(axis=1)
        elif y.ndim == 2 and y.shape[1] == 1:
            y = y.ravel()
        return pd.DataFrame({"prediction": y})

    def predict(self, context, model_input: pd.DataFrame):
        x = self._pre(model_input)
        raw = (
            self.model.predict_proba(x)
            if self.post.get("output") == "proba"
            else self.model.predict(x)
        )
        return self._post(raw)


# --- Paths and verification ---
model_name = "pyfunc_xgb"
artifact_root = get_artifact_root(catalog, schema, volume)
artifacts = get_pyfunc_xgb_paths(artifact_root)
checksums_path = get_checksums_path(artifact_root)
verify_artifacts(checksums_path, [("pyfunc_xgb.pkl", artifacts["xgb_model"]), ("pyfunc_preproc.json", artifacts["preproc"]), ("pyfunc_postproc.json", artifacts["postproc"])])
canonical_input_path = get_canonical_input_path(artifact_root)
verify_artifacts(checksums_path, [("canonical_input.json", canonical_input_path)])
canonical_input, feature_columns = load_canonical_input_json(canonical_input_path)
# Persist feature_columns so load_context can read it (PythonModelContext only has .artifacts)
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(feature_columns, f)
    _fc_path = f.name
artifacts_with_fc = {**artifacts, "feature_columns": _fc_path}


### Infer model signature

Infers input/output schema using canonical sample input (if provided).  
The inferred signature ensures consistent contract enforcement during serving and batch inference.

In [ ]:
# --- Load PyFunc context, run one prediction, infer MLflow signature ---
ctx = SimpleNamespace(artifacts=artifacts_with_fc)
model = XGBPyFunc()
model.load_context(ctx)
y_pred = model.predict(ctx, canonical_input)
signature = validate_and_infer_signature(canonical_input, y_pred)

### Log to MLflow and register to Unity Catalog

Logs the PyFunc model using `mlflow.pyfunc`. Artifacts, signature, and metadata are recorded in MLflow tracking.

Registers the logged model in Unity Catalog Model Registry.

Ensures:
- Governance
- Lineage
- Versioning

In [ ]:
# --- Log as pyfunc (artifacts copied into run), register in Unity Catalog ---
mlflow.set_registry_uri("databricks-uc")
deps = ["threadpoolctl>=3.5.0", "xgboost", "pandas", "numpy"]
with mlflow.start_run() as run:
    mlflow.pyfunc.log_model(python_model=model, artifact_path=model_name, artifacts=artifacts_with_fc, input_example=canonical_input, signature=signature, extra_pip_requirements=deps)
    model_uri = f"runs:/{run.info.run_id}/{model_name}"
registered_name = f"{catalog}.{schema}.{model_name}"
result = mlflow.register_model(model_uri=model_uri, name=registered_name)
client = MlflowClient()
register_in_uc(client, model_uri, registered_name, str(result.version))

# For job pipelines: pass version to downstream tasks (e.g. evaluation)
dbutils.jobs.taskValues.set(key="model_version", value=str(result.version))
dbutils.jobs.taskValues.set(key="model_name", value=registered_name)

# Sanity check: load the version we just registered and run predict (Champion is set later in 3_model_approval)
loaded = mlflow.pyfunc.load_model(f"models:/{registered_name}/{result.version}")
loaded.predict(canonical_input)
print(f"{registered_name} v{result.version} loaded; predict OK.")